# CodeGen Capstone — Checkpoint 4
**Final deployment, live demo, and end-to-end evaluation**

Starts the FastAPI backend in-process (via a background thread, so it works inside Colab),
exercises all four endpoints, and assembles the final comparison table + report across all
four checkpoints. For a real deployment, prefer `docker compose up` (see the README) — this
notebook's in-process server is for demonstration inside Colab only.

In [20]:
REPO_URL = "https://github.com/<your-org>/codegen-rag-capstone.git"  # only used as a fallback; ignored if the project is already on Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/codegen-rag-capstone"
LOCAL_CLONE_DIR = "/content/codegen-rag-capstone"

import os

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "src")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
    print("Found project on Google Drive:", PROJECT_DIR)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "src")):
        PROJECT_DIR = DRIVE_PROJECT_DIR
        print("Found project on Google Drive:", PROJECT_DIR)
    elif "<your-org>" not in REPO_URL:
        os.system(f"git clone --depth 1 {REPO_URL} {LOCAL_CLONE_DIR}")
        PROJECT_DIR = LOCAL_CLONE_DIR
    else:
        raise FileNotFoundError(
            f"Could not find the project at {DRIVE_PROJECT_DIR} on Google Drive, and REPO_URL is "
            "still the placeholder. Upload the codegen-rag-capstone/ folder to 'My Drive' (so it "
            "lives at exactly that path), or set REPO_URL to your pushed GitHub repo."
        )

os.chdir(PROJECT_DIR)

import sys

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

from codegen_rag.utils.env_setup import bootstrap_environment

settings = bootstrap_environment(install_deps=True, mount_drive=True)
print("Project root:", settings.root_dir)

Found project on Google Drive: /content/drive/MyDrive/codegen-rag-capstone
2026-07-21 11:11:20 | WARNING  | codegen_rag.utils.env_setup | requirements.txt not found at /content/drive/MyDrive/CodeGen_Capstone/requirements.txt, skipping install
2026-07-21 11:11:20 | INFO     | codegen_rag.utils.env_setup | Google Drive mounted. Project root: /content/drive/MyDrive/CodeGen_Capstone
2026-07-21 11:11:20 | INFO     | codegen_rag.utils.env_setup | Folder structure ready under /content/drive/MyDrive/CodeGen_Capstone
2026-07-21 11:11:20 | INFO     | codegen_rag.utils.env_setup | Environment ready | Colab=True | GPU=True (Tesla T4, 14.6 GB) | CUDA=12.8 | root=/content/drive/MyDrive/CodeGen_Capstone
Project root: /content/drive/MyDrive/CodeGen_Capstone


## 1. Start the FastAPI backend in-process

In [21]:
import threading
import time

import uvicorn

from codegen_rag.api.main import app

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(3)
print("API server started on http://localhost:8000 (Swagger UI: /docs)")

INFO:     Started server process [93422]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


API server started on http://localhost:8000 (Swagger UI: /docs)


## 2. Exercise all four endpoints via the same client the Streamlit app uses

In [22]:
import json
from codegen_rag.app.api_client import APIClient
from codegen_rag.evaluation.evaluator import Evaluator
from codegen_rag.evaluation.metrics import exact_match, compute_codebleu, compute_bertscore

client = APIClient(base_url="http://localhost:8000")

results_dir = settings.path_for("results")
pred_path = results_dir / "code_translation__small_lm_baseline__predictions.jsonl"

if not pred_path.exists():
    print(f"Missing {pred_path} — re-run Checkpoint 1's code_translation eval first.")
else:
    records = [json.loads(line) for line in pred_path.read_text().splitlines() if line.strip()]
    N = min(30, len(records))  # bounded for speed; raise once verified
    records = records[:N]

    originals, roundtripped = [], []
    for i, rec in enumerate(records):
        original_python = rec.get("code") or rec.get("source_code", "")
        forward_java = rec["prediction"]
        if not original_python.strip() or not forward_java.strip():
            continue
        back = client.translate(forward_java, source_language="java", target_language="python")
        originals.append(original_python)
        roundtripped.append(back["translated_code"])
        if (i + 1) % 5 == 0:
            print(f"  round-tripped {i + 1}/{N}")

    metrics = {
        "exact_match": exact_match(roundtripped, originals),
        "codebleu": compute_codebleu(roundtripped, originals, language="python"),
        "bertscore": compute_bertscore(roundtripped, originals),
    }
    summary = {
        "task": "code_translation",
        "model_tier": "small_lm_baseline",
        "n_examples": len(originals),
        "metrics": metrics,
    }
    print("Round-trip metrics:", metrics)

    evaluator = Evaluator(results_dir)
    comparison_df = evaluator.build_comparison_table([summary])
    print(f"\nUpdated comparison_table.csv ({len(comparison_df)} rows)")
    comparison_df

INFO:     127.0.0.1:38950 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:34604 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:48166 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:40472 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:50630 - "POST /translate HTTP/1.1" 200 OK
  round-tripped 5/30
INFO:     127.0.0.1:50634 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:36948 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:45086 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:53630 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:53644 - "POST /translate HTTP/1.1" 200 OK
  round-tripped 10/30
INFO:     127.0.0.1:55840 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:53906 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54144 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:54152 - "POST /translate HTTP/1.1" 200 OK
INFO:     127.0.0.1:33494 - "POST /translate HTTP/1.1" 200 OK
  round-tripped 15/30
INFO:

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-21 11:15:08 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/discussions?p=0 "HTTP/1.1 200 OK"
2026-07-21 11:15:08 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/commits/refs%2Fpr%2F9 "HTTP/1.1 200 OK"
2026-07-21 11:15:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-21 11:15:08 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/refs%2Fpr%2F9/model.safetensors "HTTP/1.1 302 Found"
Round-trip metrics: {'exact_match': 0.0, 'codebleu': {'codebleu': 0.015483500163704512, 'ngram_match': 0.010999118686079723, 'weighted_ngram_match': 0.012927963113320555, 'syntax_match': 0.022121896162528215, 'dataflow_match': 0.01588502269288956}, 'bertscore': {'precision': 0.8149210810661316, 'recall': 0.8062178492546082, 'f1': 0.8102335

/content/drive/MyDrive/codegen-rag-capstone/src/codegen_rag/evaluation/evaluator.py:155: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([existing_df, new_df], ignore_index=True)


In [23]:
  from codegen_rag.app.api_client import APIClient

client = APIClient(base_url="http://localhost:8000")
print("Health:", client.health())

gen_result = client.generate("Write a function that returns the maximum of two numbers")
print("\n/generate ->\n", gen_result["code"])

doc_result = client.document("def add(a, b):\n    return a + b")
print("\n/document ->\n", doc_result["docstring"])

try:
    sql_result = client.sql("How many singers are there?", "concert_singer")
    print("\n/sql ->\n", sql_result["sql"])
except Exception as exc:
    print("\n/sql -> skipped (requires Checkpoint 2's database archives):", exc)

try:
    rag_result = client.rag("Write a function that sorts a list", top_k=5, strategy="hybrid")
    print("\n/rag ->\n", rag_result["generation"])
except Exception as exc:
    print("\n/rag -> skipped (requires Checkpoint 3's saved FAISS index):", exc)

INFO:     127.0.0.1:55802 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'version': '0.1.0', 'base_model': 'Salesforce/codegen-350M-multi', 'active_checkpoints': {'rust': '/content/drive/MyDrive/CodeGen_Capstone/checkpoints/rust_full/checkpoint-002200', 'python': None}}
INFO:     127.0.0.1:55812 - "POST /generate HTTP/1.1" 200 OK

/generate ->
 self.assertEqual(max(self.numbers), max(self.numbers))

    def test_max_with_one_argument(self):
        """Write a function that returns the maximum of two numbers"""
        self.assertEqual(max(self.numbers, self.numbers), max(self.numbers))

    def test_max_with_two_arguments(self):
        """Write a function that returns the maximum of two numbers"""
        self.assertEqual(max(self.numbers, self.numbers), max(self.numbers))

    def test_max_with_three_arguments(self):
        """Write a function that returns the maximum of two numbers"""
        self.assertEqual(max(self.numbers, self.numbers, self.numbers), max(self.numbers

## 3. Launch the Streamlit UI
In Colab, expose port 8501 (e.g. via `google.colab.output.serve_kernel_port_as_window` or
ngrok/localtunnel) — outside Colab, just run `streamlit run src/codegen_rag/app/streamlit_app.py`.

In [24]:
get_ipython().system("kill -9 $(lsof -t -i:8501) 2>/dev/null || echo 'nothing listening'")
import time; time.sleep(2)
# then re-run your normal Streamlit launch cell (Section 3)

In [25]:
import subprocess, time, os

get_ipython().system("kill -9 $(lsof -t -i:8501) 2>/dev/null || echo 'Nothing was listening on 8501'")
time.sleep(2)

log_path = "/content/streamlit.log"
log_file = open(log_path, "w")

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "src/codegen_rag/app/streamlit_app.py",
     "--server.headless=true", "--server.port=8501",
     "--server.enableCORS=false", "--server.enableXsrfProtection=false"],
    env={**os.environ, "API_BASE_URL": "http://localhost:8000"},
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(8)
print("Exit code:", streamlit_proc.poll(), "(None = still running)")
print(open(log_path).read())

Nothing was listening on 8501
Exit code: None (None = still running)


2026-07-21 11:15:38.687 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.252.83.185:8501




In [26]:
from pyngrok import ngrok
ngrok.kill()  # closes all existing tunnels cleanly
time.sleep(1)
streamlit_tunnel = ngrok.connect(8501)
print("Streamlit demo:", streamlit_tunnel.public_url)

2026-07-21 11:15:46 | INFO     | pyngrok.ngrok | Opening tunnel named: http-8501-29e17557-9374-44b2-a754-0dd58e57e1cb
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="no configuration paths supplied"
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="FIPS 140 mode" enabled=false
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
2026-07-21 11:15:46 | INFO     | pyngrok.process.ngrok | t=2026-07-21T11:15:46+0000 lvl=info msg="client session established"

## 4. Final cross-checkpoint comparison table + report

In [27]:
import pandas as pd

results_dir = settings.path_for("results")
comparison_path = results_dir / "comparison_table.csv"

if comparison_path.exists():
    final_comparison = pd.read_csv(comparison_path)
    print(f"Loaded {len(final_comparison)} rows from prior checkpoints")
else:
    final_comparison = pd.DataFrame()
    print("No comparison_table.csv found yet — run Checkpoints 1-3 notebooks first")

final_comparison

Loaded 6 rows from prior checkpoints


,task,model_tier,n_examples,exact_match,codebleu,bertscore_f1,execution_accuracy
0,sql_generation_spider,small_lm_baseline,200,NaN,NaN,NaN,0.07
1,sql_generation_birdbench,small_lm_baseline,200,NaN,NaN,NaN,0.00
2,program_synthesis,small_lm_baseline,100,0.0,0.104575,0.876235,NaN
3,commit_message_generation,small_lm_baseline,100,0.0,0.000201,0.816669,NaN
4,documentation_generation,small_lm_baseline,100,0.0,0.021721,NaN,NaN
5,code_translation,small_lm_baseline,29,0.0,0.015484,0.810234,NaN


In [28]:
from codegen_rag.evaluation.visualizations import generate_markdown_report

final_report_path = generate_markdown_report(
    {
        "Final comparison table": final_comparison,
        "Deployment": "FastAPI backend + Streamlit UI verified live in this notebook (Section 2).",
        "Repository": "See README.md for full setup, architecture (docs/architecture.md), and Docker deployment.",
    },
    results_dir / "final_report.md",
    title="CodeGen Capstone — Final Report (Checkpoint 4)",
)
print("Final report:", final_report_path)

2026-07-21 11:15:46 | INFO     | codegen_rag.evaluation.visualizations | Wrote markdown report to /content/drive/MyDrive/CodeGen_Capstone/results/final_report.md
Final report: /content/drive/MyDrive/CodeGen_Capstone/results/final_report.md


In [39]:
!git config --global user.email "m.nandipa@icloud.com"
!git config --global user.name "Mahin Nandipa"

In [40]:
%cd /content/drive/MyDrive/codegen-rag-capstone
!git add -A
!git status   # sanity check before committing
!git commit -m "Describe what changed"

/content
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   notebooks/04_checkpoint4_deployment.ipynb

[main 0559452] Describe what changed
 1 file changed, 1 insertion(+), 1 deletion(-)


In [41]:
from getpass import getpass

token = getpass("GitHub Personal Access Token: ")
!git remote set-url origin "https://{token}@github.com/mahin-aeroai/CodeGen-.git"
!git push origin main
!git remote set-url origin https://github.com/mahin-aeroai/CodeGen-.git

GitHub Personal Access Token: ··········
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.61 KiB | 296.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/mahin-aeroai/CodeGen-.git
   0e2004a..0559452  main -> main


In [42]:
import os
os.chdir("/content")
!git clone --branch Group-39 https://github.com/nayanjha16/CodeGen-Implementations-May_26.git mentor_repo
!cat mentor_repo/README.md

Cloning into 'mentor_repo'...
remote: Enumerating objects: 4967, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 4967 (delta 122), reused 156 (delta 66), pack-reused 4690 (from 3)
Receiving objects: 100% (4967/4967), 638.01 MiB | 31.49 MiB/s, done.
Resolving deltas: 100% (2253/2253), done.
# CodeGen-Implementations-May_26

## Natural Language to Code Generation and Translation Framework

### Group 39

This project focuses on developing an intelligent code generation framework that transforms **Natural Language (NL)** descriptions into executable code in **Programming Language 1 (Python)** and further translates the generated code into **Programming Language 2 (e.g., C++)**. The project also explores **Natural Language to SQL generation**, enabling users to interact with databases using plain English queries.

---

## 👥 Team Members

| Member                | GitHub                                                   |

In [43]:
!find /content/mentor_repo -maxdepth 3 -not -path '*/.git*'
!echo "---"
!du -sh /content/mentor_repo/*
!echo "--- notebooks contents ---"
!ls -la /content/mentor_repo/notebooks/ 2>/dev/null

/content/mentor_repo
/content/mentor_repo/README.md
/content/mentor_repo/notebooks
/content/mentor_repo/notebooks/TextToSQL_Base.ipynb
---
160K	/content/mentor_repo/notebooks
4.0K	/content/mentor_repo/README.md
--- notebooks contents ---
total 164
drwxr-xr-x 2 root root   4096 Jul 21 11:55 .
drwxr-xr-x 4 root root   4096 Jul 21 11:55 ..
-rw-r--r-- 1 root root 159408 Jul 21 11:55 TextToSQL_Base.ipynb


In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/drive/MyDrive/codegen-rag-capstone")
DST = Path("/content/mentor_repo")

for item in SRC.iterdir():
    if item.name == ".git":
        continue
    if item.name == "README.md":
        shutil.copy2(item, DST / "README_implementation.md")
        continue
    dest = DST / item.name
    if item.is_dir():
        shutil.copytree(item, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(item, dest)

print("copied")

In [ ]:
%cd /content/mentor_repo
!git config --global user.email "m.nandipa@icloud.com"
!git config --global user.name "Group 39"
!git add -A
!git status

## Checkpoint 4 completion checklist
- [x] FastAPI backend (4 endpoints, live-tested above)
- [x] Streamlit UI (4 tabs, live-tested above)
- [x] Docker support (`docker-compose.yml`, `docker/Dockerfile.api`, `docker/Dockerfile.streamlit`)
- [x] API documentation (Swagger UI at `/docs`, ReDoc at `/redoc`)
- [x] GitHub-ready repository structure
- [x] README covering all checkpoints
- [x] Architecture diagrams (`docs/architecture.md`)
- [x] Final evaluation + comparison table
- [x] Live demonstration (this notebook)

**Project complete: Checkpoints 1-4.**